In [ ]:
!pip install torch rdkit pandas scikit-learn torch_geometric


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split


In [ ]:
def atom_features(atom):
    return torch.tensor([
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetHybridization()),
        atom.GetIsAromatic()
    ], dtype=torch.float)

def bond_features(bond):
    return torch.tensor([
        bond.GetBondTypeAsDouble(),
        bond.IsInRing()
    ], dtype=torch.float)

def smiles_to_graph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumBonds() == 0:
        return None
    x = torch.stack([atom_features(atom) for atom in mol.GetAtoms()])
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]
        edge_attr  += [bond_features(bond), bond_features(bond)]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr  = torch.stack(edge_attr)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([y], dtype=torch.float))


In [ ]:
class GNNRegressor(nn.Module):
    def __init__(self, node_feat_dim, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(node_feat_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin1  = nn.Linear(hidden_dim, 64)
        self.lin2  = nn.Linear(64, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = torch.relu(self.conv1(x, edge_index))
        x = torch.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        x = torch.relu(self.lin1(x))
        # squeeze [B, 1] -> [B] to match batch.y; otherwise MSELoss broadcasts to [B, B]
        return self.lin2(x).squeeze(-1)


In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(batch), batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds.append(model(batch).cpu())
            truths.append(batch.y.cpu())
    return torch.cat(preds), torch.cat(truths)


In [ ]:
dataset_path = "/content/drive/MyDrive/Projects/NeurIPS Polymer Prediction 2025/data/train.csv"
models_dir   = "/content/drive/MyDrive/Projects/NeurIPS Polymer Prediction 2025/models"
os.makedirs(models_dir, exist_ok=True)

device  = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
loss_fn = nn.MSELoss()

# (target_column, test_size, save_filename)
TARGETS = [
    ("Rg",      0.2, "gnn_best_rg_model.pth"),
    ("Tc",      0.2, "gnn_best_tc_model.pth"),
    ("Density", 0.1, "gnn_best_density_model.pth"),
    ("FFV",     0.1, "gnn_best_ffv_model.pth"),
]

full_df = pd.read_csv(dataset_path)
trained_models = {}

for target, test_size, filename in TARGETS:
    sep = "=" * 50
    print(f"\n{sep}\nTraining GNN for: {target}\n{sep}")

    df = full_df[full_df["SMILES"].notna() & full_df[target].notna()]
    graphs = [g for g in (smiles_to_graph(r["SMILES"], r[target]) for _, r in df.iterrows()) if g is not None]
    print(f"{len(graphs)} valid graphs ({len(df) - len(graphs)} skipped)")

    train_graphs, test_graphs = train_test_split(graphs, test_size=test_size, random_state=42)
    train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
    test_loader  = DataLoader(test_graphs,  batch_size=32)

    model     = GNNRegressor(node_feat_dim=train_graphs[0].x.size(1)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    best_mse  = float("inf")
    model_path = f"{models_dir}/{filename}"

    for epoch in range(1000):
        loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        preds, truths = evaluate(model, test_loader, device)
        mse = torch.mean((preds - truths) ** 2).item()
        print(f"Epoch {epoch:03d} | Train Loss: {loss:.4f} | Test MSE: {mse:.4f}")
        if mse < best_mse:
            best_mse = mse
            torch.save(model.state_dict(), model_path)
            print(f"  -> New best saved (MSE: {mse:.4f})")

    # keep the best checkpoint, not the last epoch's weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    trained_models[target] = model
    print(f"Finished {target}. Best MSE: {best_mse:.4f}")
